# Browser GUI in Colab

Run the sections **in order**. Each section says when to run it.

| Section | When to run |
|---|---|
| 1. Install | Once per new Colab session |
| 2. Setup | Once per new Colab session |
| 3. Start | To launch the browser |
| 4. Connect | To see the browser (inside the notebook) |
| 5. Save downloads to Drive | After you download something |
| 6. Stop | When you are finished |

Downloads made in the browser land in `/content/downloads`. Copy them to Drive in section 5 before the session ends.


## 1. Install (once per session)

In [1]:
%%bash
apt-get update -qq
apt-get install -y -qq xvfb x11vnc novnc websockify fluxbox
wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
apt-get install -y -qq ./google-chrome-stable_current_amd64.deb
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
echo "Install done"


Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../00-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package poppler-data.
Preparing to unpack .../01-poppler-data_0.4.12-1_all.deb ...
Unpacking poppler-data (0.4.12-1) ...
Selecting previously unselected package python3-netifaces:amd64.
Preparing to unpack .../02-python3-netifaces_0.11.0-2build3_amd64.deb ...
Unpacking python3-netifaces:amd64 (0.11.0-2build3) ...
Selecting previously unselected package python3-pygments.
Preparing to unpack .../03-python3-pygments_2.17.2+dfsg-1_all.deb ...
Unpacking python3-pygments (2.17.2+dfsg-1) ...
Selecting previously unselected package python3-yaml.
Preparing to unpack .../04-python3-yaml_6.0.1-2build2_amd64.deb ...
Unpacking python3-yaml (6.0.1-2build2) ...
Selecting previously unselected packa

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Setup (once per session)
Mounts Drive and sets Chrome's download folder to `/content/downloads`.

In [ ]:
import os, json, subprocess, time, re
from google.colab import drive

drive.mount('/content/drive')

DOWNLOADS = '/content/downloads'
PROFILE = '/content/chrome-profile'
DRIVE_DEST = '/content/drive/MyDrive/browser_downloads'   # change if you want another folder

os.makedirs(DOWNLOADS, exist_ok=True)
os.makedirs(f'{PROFILE}/Default', exist_ok=True)
os.makedirs(DRIVE_DEST, exist_ok=True)

prefs = {"download": {"default_directory": DOWNLOADS, "prompt_for_download": False}}
json.dump(prefs, open(f'{PROFILE}/Default/Preferences', 'w'))
print("Setup done")


## 3. Start
Starts the virtual screen, VNC, noVNC and Chrome. Safe to re-run: it stops old processes first.

In [ ]:
def stop_all():
    for name in ['google-chrome', 'chrome', 'x11vnc', 'websockify', 'fluxbox', 'Xvfb', 'cloudflared']:
        subprocess.run(['pkill', '-f', name], stderr=subprocess.DEVNULL)
    subprocess.run('rm -f /tmp/.X1-lock', shell=True)
    time.sleep(2)

stop_all()

env = dict(os.environ, DISPLAY=':1')
def run(cmd):
    subprocess.Popen(cmd, env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

run(['Xvfb', ':1', '-screen', '0', '1280x720x24'])
time.sleep(2)
run(['fluxbox'])
run(['x11vnc', '-display', ':1', '-nopw', '-forever', '-shared', '-rfbport', '5900'])
run(['websockify', '--web', '/usr/share/novnc', '6080', 'localhost:5900'])
run(['google-chrome', '--no-sandbox', '--disable-dev-shm-usage',
     f'--user-data-dir={PROFILE}', '--start-maximized'])
time.sleep(3)

# quick check
n = subprocess.run('ps aux | grep -E "Xvfb|x11vnc|websockify|chrome" | grep -v grep | wc -l',
                   shell=True, capture_output=True, text=True).stdout.strip()
web = subprocess.run('curl -sI http://localhost:6080/vnc.html | head -1',
                     shell=True, capture_output=True, text=True).stdout.strip()
print(f"Processes running: {n}")
print(f"noVNC status: {web or 'not responding'}")


## 4. Connect
Shows the browser inside the notebook. Use **4a** first. Use **4b** only if 4a fails.

### 4a. Inside the notebook (no tunnel)

In [4]:
from google.colab import output
output.serve_kernel_port_as_iframe(6080, path='/vnc.html?autoconnect=true&resize=scale', height='700')


<IPython.core.display.Javascript object>

## 5. Save downloads to Drive
Run after downloading files in the browser.

In [ ]:
import shutil, glob
files = [f for f in glob.glob(f'{DOWNLOADS}/*') if not f.endswith('.crdownload')]
if not files:
    print("No finished downloads yet in", DOWNLOADS)
for f in files:
    dest = os.path.join(DRIVE_DEST, os.path.basename(f))
    shutil.copy2(f, dest)
    print("Copied:", os.path.basename(f), "->", DRIVE_DEST)


In [ ]:
# Optional: see what is in the download folder (files ending in .crdownload are still downloading)
!ls -lh /content/downloads


## 6. Stop
Stops the browser, VNC and tunnel. Run **section 5 first** if you have unsaved downloads.

In [ ]:
stop_all()
left = subprocess.run('ps aux | grep -E "Xvfb|x11vnc|websockify|chrome|fluxbox|cloudflared" | grep -v grep | wc -l',
                      shell=True, capture_output=True, text=True).stdout.strip()
print(f"Stopped. Processes still running: {left}")
